# Glacier outputs postprocessing

In [ ]:
# spatial
import geopandas as gpd
import pandas as pd
import xarray as xr 
from pathlib import Path
from tqdm import tqdm

from src import glaciers
from src.utils import load_config
cfg = load_config()

In [ ]:
# lookup tables
andeangc_lookup   = gpd.read_file(cfg["dir"]["andeangc"] + "/AndeanGC_shape.gpkg").set_index('gauge_id')[["geometry"]]
hydroatlas_lookup = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index('gauge_id')[["geometry"]]

glaciers_lookup = pd.concat([gpd.read_file(cfg["dir"]["data"] + "/glacier_outlines/RGI60_16.shp").set_index('RGIId')[["geometry"]], 
                             gpd.read_file(cfg["dir"]["data"] + "/glacier_outlines/RGI60_17.shp").set_index('RGIId')[["geometry"]]])

## Glacier-wide aggregation

In [ ]:
# period 2000-2025
columns = ["rgi_subregion", "cenlat", "cenlon", "rgi_area_km2", "rgi_year", "inv_volume_km3"]
glacier_metadata = pd.read_csv(cfg["dir"]["pipeline"] + "/oggm/glacier_statistics.csv", low_memory=False, index_col=0)
glacier_metadata = glacier_metadata[columns]

df_historical = xr.open_dataset(cfg["dir"]["pipeline"] + "/oggm/run_output_historical.nc").volume.load()
df_historical = df_historical.drop_vars(["hydro_month", "hydro_year", "calendar_year", "calendar_month"])

# A glacier is kept only if it ran in every simulation (historical, all future and all what-if
# runs), so that all runs cover the same glaciers. A failed run is all NaN. A what-if run that
# failed because the glacier had already vanished in its future run (zero volume in 2100, e.g.
# lake-terminating glaciers, for which OGGM cannot set a water level) is not a failure: its melt is zero.
failed = df_historical.isnull().all("time").to_pandas()

df_future = []
files = list(Path(cfg["dir"]["pipeline"]).glob("oggm/*output*"))
files = [f for f in files if "historical" not in f.name]

# period 2000-2099
for gcm in files:
    rid = gcm.name.removeprefix("run_output_").removesuffix(".nc")
    ts_future = xr.open_dataset(gcm).volume.load()
    ts_whatif = xr.open_dataset(gcm.parent / f"run_output_historical_{rid}.nc").volume.load()
    vanished = (ts_future.sel(time=2100) == 0).to_pandas()
    failed |= ts_future.isnull().all("time").to_pandas().reindex(failed.index, fill_value=True)
    failed |= (ts_whatif.isnull().all("time").to_pandas() & ~vanished).reindex(failed.index, fill_value=True)

    ts_future = ts_future.drop_vars(["hydro_month", "hydro_year", "calendar_year", "calendar_month"])
    ts_future = ts_future.sel(time = 2100).to_pandas().rename("volume_2100_{}".format(gcm.name.split('_')[2]))
    df_future.append(ts_future)

df_future = pd.concat(df_future, axis=1)
df_future = df_future.T.groupby(level=0).mean().T
df_historical = df_historical.sel(time = 2020).to_pandas().rename("volume_2020_historical")
df = pd.concat([glacier_metadata, df_historical, df_future], axis = 1)
df = df[~failed.reindex(df.index, fill_value=True)] # remove glaciers that failed in any OGGM run
df.to_csv("data/glacier/glacier_metadata_evolution.csv")

# list with glaciers that successfully ran in OGGM
glacier_sucess = df.index
area = glacier_metadata.rgi_area_km2
print(f"Glaciers removed: {len(area) - len(df)} ({100 * (1 - area[df.index].sum() / area.sum()):.2f}% of area)")

## Region-wide aggregation

In [ ]:
# period 2000-2025 (historical)
df_historical = xr.open_dataset(cfg["dir"]["pipeline"] + "/oggm/run_output_historical.nc")[["melt_on_glacier", "volume"]]
df_historical = df_historical.drop_vars(["hydro_month", "calendar_month", "hydro_year", "calendar_year"])
df_historical = df_historical.sel(rgi_id=glacier_sucess)
df_historical.to_netcdf("data/glacier/glacier_evolution_2000_2025_glacier.nc")
df_historical.sum(dim="rgi_id").to_netcdf("data/glacier/glacier_evolution_2000_2025.nc")

# period 2026-2099
files = list(Path(cfg["dir"]["pipeline"]).glob("oggm/*output*"))
files = [f for f in files if "historical" not in f.name]
warming_levels = [f.name.split('_')[2] for f in files]

df_future = []
for gcm in files:
    ts_future = xr.open_dataset(gcm)[["melt_on_glacier", "volume"]].load()
    ts_future = ts_future.drop_vars(["hydro_month", "calendar_month", "hydro_year", "calendar_year"])
    ts_future = ts_future.sel(rgi_id=glacier_sucess)
    df_future.append(ts_future)

df_future = xr.concat(df_future, dim="warming_level")
df_future = df_future.assign_coords(warming_level=("warming_level", warming_levels))
df_future.sum(dim="rgi_id").to_netcdf("data/glacier/glacier_evolution_2026_2099.nc")

In [ ]:
# period 2000-2025 (what-if)
files = list(Path(cfg["dir"]["pipeline"]).glob("oggm/*output_historical_*"))
warming_levels = [f.name.split('_')[3] for f in files]

df_future = []
for gcm in files:
    ts_future = xr.open_dataset(gcm)[["melt_on_glacier", "volume"]].load()
    ts_future = ts_future.drop_vars(["hydro_month", "calendar_month", "hydro_year", "calendar_year"])
    ts_future = ts_future.sel(rgi_id=glacier_sucess)
    df_future.append(ts_future)

df_future = xr.concat(df_future, dim="warming_level")
df_future = df_future.assign_coords(warming_level=("warming_level", warming_levels))
df_future.to_netcdf("data/glacier/glacier_evolution_2000_2025_whatif.nc")

## Basin-wide aggregation

In [ ]:
hydroatlas_lookup = glaciers.calculate_basin_overlap(glaciers_lookup, hydroatlas_lookup)
andeangc_lookup   = glaciers.calculate_basin_overlap(glaciers_lookup, andeangc_lookup)

In [ ]:
paths = list(Path(cfg["dir"]["pipeline"]).glob("oggm/*output_historical*nc"))

for path in tqdm(paths):  

    # Load and preprocess data once
    oggm_ts = xr.open_dataset(path)[glaciers.variables].load()
    oggm_ts = oggm_ts.sel(time = slice(int(cfg["time"]["ref_start"][:4]), int(cfg["time"]["ref_end"][:4])))
    oggm_ts = oggm_ts.sel(rgi_id=glacier_sucess)
    oggm_ts = glaciers.change_calendar(oggm_ts)
    oggm_ts = glaciers.hydro_variables(oggm_ts, only_melt=True)
    oggm_ts = oggm_ts.fillna(0)  # what-if runs of glaciers that had vanished (see glacier_sucess)

    # for HydroATLAS and AndeanGC
    hydroatlas_lookup = hydroatlas_lookup[hydroatlas_lookup.index.isin(oggm_ts.rgi_id.values)]
    oggm_ts_hydroatlas = glaciers.aggregate_data_by_basin(oggm_ts, hydroatlas_lookup)
    andeangc_lookup = andeangc_lookup[andeangc_lookup.index.isin(oggm_ts.rgi_id.values)]
    oggm_ts_andeangc = glaciers.aggregate_data_by_basin(oggm_ts, andeangc_lookup)
    
    # Merge and save
    oggm_ts = xr.merge([oggm_ts_andeangc, oggm_ts_hydroatlas], join='outer', compat='no_conflicts').astype('float32')
    oggm_ts.to_netcdf("data/glacier/" + path.name, encoding={var: {"compression": "zstd"} for var in oggm_ts.data_vars})